# QF602 HW 6 - Breeden-Litzenberger Static Replication

**Parameters:** $r = q = 0$, $S_0 = 1$, $T = 4$

**Implied volatility surface:**
$$\Sigma(K) = 0.510 - 0.591K + 0.376K^2 - 0.105K^3 + 0.011K^4$$
with $\Sigma(K) = \Sigma(3)$ for $K > 3$.

**Breeden-Litzenberger formula:**
$$V_0 = e^{-rT} V_T(F_0(T)) + \int_0^{F_0(T)} Put(K,T) \frac{\partial^2 V_T(K)}{\partial K^2} dK + \int_{F_0(T)}^{\infty} Call(K,T) \frac{\partial^2 V_T(K)}{\partial K^2} dK$$

## Requisite Libraries and Functions

In [1]:
## Import Libraries

import numpy as np
from scipy.stats import norm
from scipy.integrate import quad

In [2]:
# Parameters
r = 0.0
q = 0.0
S0 = 1.0
T = 4.0
F0 = S0 * np.exp((r - q) * T)  # = 1.0

print(f"Forward price F0(T) = {F0}")

Forward price F0(T) = 1.0


In [3]:
## Functions to Calculate Payoffs

def implied_vol(K):
    """Implied volatility surface Sigma(K)"""
    K = np.atleast_1d(np.array(K, dtype=float))
    sigma = 0.510 - 0.591*K + 0.376*K**2 - 0.105*K**3 + 0.011*K**4
    sigma_cap = implied_vol_scalar(3.0)
    sigma = np.where(K > 3.0, sigma_cap, sigma)
    return sigma if sigma.size > 1 else float(sigma)

def implied_vol_scalar(K):
    """Scalar version"""
    if K > 3.0:
        K = 3.0
    return 0.510 - 0.591*K + 0.376*K**2 - 0.105*K**3 + 0.011*K**4

# Verify vol at a few strikes
for K in [0.5, 1.0, 2.0, 3.0, 4.0]:
    print(f"Sigma({K}) = {implied_vol_scalar(K):.4f}")

Sigma(0.5) = 0.2961
Sigma(1.0) = 0.2010
Sigma(2.0) = 0.1680
Sigma(3.0) = 0.1770
Sigma(4.0) = 0.1770


In [4]:
## Black Formulae

def bs_call(S, K, r, q, T, sigma):
    """Black-Scholes call price"""
    if sigma <= 0 or K <= 0:
        return max(S * np.exp(-q*T) - K * np.exp(-r*T), 0.0)
    F = S * np.exp((r - q) * T)
    d1 = (np.log(F/K) + 0.5 * sigma**2 * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return np.exp(-r*T) * (F * norm.cdf(d1) - K * norm.cdf(d2))

def bs_put(S, K, r, q, T, sigma):
    """Black-Scholes put price"""
    if sigma <= 0 or K <= 0:
        return max(K * np.exp(-r*T) - S * np.exp(-q*T), 0.0)
    F = S * np.exp((r - q) * T)
    d1 = (np.log(F/K) + 0.5 * sigma**2 * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return np.exp(-r*T) * (K * norm.cdf(-d2) - F * norm.cdf(-d1))

def Call(K):
    sigma = implied_vol_scalar(K)
    return bs_call(S0, K, r, q, T, sigma)

def Put(K):
    sigma = implied_vol_scalar(K)
    return bs_put(S0, K, r, q, T, sigma)

# Quick check: put-call parity at K=1
print('Quick Check for Put-Call Parity')
print(f"Call(1.0) = {Call(1.0):.6f}")
print(f"Put(1.0)  = {Put(1.0):.6f}")
print(f"C - P = S0 - K*exp(-rT) = {Call(1.0) - Put(1.0):.6f} is {Call(1.0) - Put(1.0) == S0 - 1*np.exp(-r*T)}!")

Quick Check for Put-Call Parity
Call(1.0) = 0.159301
Put(1.0)  = 0.159301
C - P = S0 - K*exp(-rT) = 0.000000 is True!


## Payoff 1

$V_T(S_T) = \sqrt{S_T}$

Second derivative: $\frac{\partial^2 V_T}{\partial K^2} = \frac{\partial^2 \sqrt{K}}{\partial K^2} = -\frac{1}{4} K^{-3/2}$

In [5]:
def payoff1(K):
    return np.sqrt(K)

def d2_payoff1(K):
    """Second derivative of sqrt(K)"""
    return -0.25 * K**(-1.5)

# Breeden-Litzenberger formula
# V0 = e^{-rT} * V_T(F0) + integral_0^{F0} Put(K) * V''(K) dK + integral_{F0}^inf Call(K) * V''(K) dK

# First term
term0 = np.exp(-r*T) * payoff1(F0)
print(f"First term e^(-rT)*V(F0) = {term0:.6f}")

# Put integral from 0 to F0; Set lower bound as 1e-3 since integration from zero is indivisible
eps = 1e-3
put_integral, put_err = quad(lambda K: Put(K) * d2_payoff1(K), eps, F0, limit=200)
print(f"Put integral = {put_integral:.6f}  (error est: {put_err:.2e})")

# Call integral from F0 to infinity
K_max = 1e6
call_integral, call_err = quad(lambda K: Call(K) * d2_payoff1(K), F0, K_max, limit=200)
print(f"Call integral = {call_integral:.6f}  (error est: {call_err:.2e})")

V0_payoff1 = term0 + put_integral + call_integral
print(f"\nV0 for sqrt(S_T) = {V0_payoff1:.6f}")

First term e^(-rT)*V(F0) = 1.000000
Put integral = -0.018119  (error est: 9.78e-09)
Call integral = -0.000000  (error est: 9.74e-89)

V0 for sqrt(S_T) = 0.981881


In [6]:
# Sanity check: if vol were flat (ATM vol), compare with analytical
# For flat vol sigma, E[sqrt(S_T)] = S0^0.5 * exp(-sigma^2*T/8)
sigma_atm = implied_vol_scalar(1.0)
analytic_flat = np.sqrt(S0) * np.exp(-sigma_atm**2 * T / 8)
print(f"Flat-vol analytical approximation (sigma={sigma_atm:.4f}): {analytic_flat:.6f}")
print(f"(This differs from BL result due to smile)")

Flat-vol analytical approximation (sigma=0.2010): 0.980002
(This differs from BL result due to smile)


## Payoff 2: $V_T(S_T) = S_T^3$

Second derivative: $\frac{\partial^2 K^3}{\partial K^2} = 6K$

In [7]:
def payoff2(K):
    return K**3

def d2_payoff2(K):
    """Second derivative of K^3"""
    return 6.0 * K

# First term
term0_p2 = np.exp(-r*T) * payoff2(F0)
print(f"First term e^(-rT)*V(F0) = {term0_p2:.6f}")

# Put integral from 0 to F0
put_integral_p2, put_err_p2 = quad(lambda K: Put(K) * d2_payoff2(K), 0, F0, limit=200)
print(f"Put integral = {put_integral_p2:.6f}  (error est: {put_err_p2:.2e})")

# Call integral from F0 to infinity
# Set K_max as 5000 because higher values give inaccurate results (K^3 grows fast, but calls decay faster due to BS formula)
K_max = 5000
call_integral_p2, call_err_p2 = quad(lambda K: Call(K) * d2_payoff2(K), F0, K_max, limit=200)
print(f"Call integral = {call_integral_p2:.6f}  (error est: {call_err_p2:.2e})")

V0_payoff2 = term0_p2 + put_integral_p2 + call_integral_p2
print(f"\nV0 for S_T^3 = {V0_payoff2:.6f}")

First term e^(-rT)*V(F0) = 1.000000
Put integral = 0.195395  (error est: 1.77e-09)
Call integral = 0.327664  (error est: 3.41e-09)

V0 for S_T^3 = 1.523059


In [8]:
# Sanity check for S_T^3 with flat vol:
# E[S_T^3] = S0^3 * exp(3*(r-q)*T + 3*2/2*sigma^2*T) = exp(3*sigma_atm^2*T)
analytic_flat_p2 = S0**3 * np.exp(3*2*sigma_atm**2*T / 2)  # = exp(6*sigma^2*T) when r=0
print(f"Flat-vol analytical approx for S_T^3 (sigma={sigma_atm:.4f}): {analytic_flat_p2:.6f}")
print(f"BL result (with smile): {V0_payoff2:.6f}")

Flat-vol analytical approx for S_T^3 (sigma=0.2010): 1.623870
BL result (with smile): 1.523059


## Summary

In [9]:
# Summary
print("=" * 50)
print("SUMMARY")
print("=" * 50)
print(f"V0 for payoff sqrt(S_T):  {V0_payoff1:.6f}")
print(f"V0 for payoff S_T^3:      {V0_payoff2:.6f}")

SUMMARY
V0 for payoff sqrt(S_T):  0.981881
V0 for payoff S_T^3:      1.523059
